# CIFAR-10 Image Classification — CNN
**Artificial Intelligence diploma | Grecia Barragan**

The goal is to classify ten object categories using a convolutional neural network built with PyTorch. The training pipeline and model architecture follow the coursework implementation.

## Problem and context
CIFAR-10 contains 60,000 32×32 color images in ten categories (airplane, automobile, bird, cat, deer, dog, frog, horse, ship and truck). A CNN learns local visual patterns through convolution and pooling layers.

## Implementation
The network uses three convolutional layers (32, 64, 128 filters), max pooling, two fully connected layers, Adam, and cross-entropy loss. Image augmentation is applied to training data only; test preprocessing is deterministic. Execute the code below to download CIFAR-10, train the model and display an example prediction.

In [ ]:
# LIBRERIAS
# PyTorch: arquitectura y entrenamiento.
# torchvision: CIFAR-10 y transformaciones de imagenes.
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# Clases de CIFAR-10
clases = [
    'Avión', 'Automóvil', 'Pájaro', 'Gato', 'Ciervo',
    'Perro', 'Rana', 'Caballo', 'Barco', 'Camión'
]

# Aumentos aleatorios solo durante el entrenamiento.
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# El conjunto de datos se descarga localmente y no se incluye en este repositorio.
trainset = datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transform)
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)

testset = datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transform)
testloader = DataLoader(testset, batch_size=64, shuffle=False)

# Arquitectura de la red convolucional
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        x = self.pool(nn.functional.relu(self.conv1(x)))
        x = self.pool(nn.functional.relu(self.conv2(x)))
        x = self.pool(nn.functional.relu(self.conv3(x)))
        x = x.view(-1, 128 * 4 * 4)
        x = nn.functional.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Entrenamiento con Adam y cross entropy
model = CNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 10
for epoch in range(epochs):
    running_loss = 0.0
    for images, labels in trainloader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f'Epoch [{epoch + 1}/{epochs}], Loss: {running_loss / len(trainloader):.4f}')

# Guardar pesos aprendidos
torch.save(model.state_dict(), 'modelo_cifar10.pth')

# Evaluar en el conjunto de prueba sin transformaciones aleatorias.
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in testloader:
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Precisión en el conjunto de prueba: {100 * correct / total:.2f}%')

# Función auxiliar para preprocesar una imagen individual
def cargar_y_preprocesar_imagen(ruta):
    imagen = Image.open(ruta).convert('RGB')
    transform = transforms.Compose([
        transforms.Resize((32, 32)),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])
    return transform(imagen).unsqueeze(0)

# Cargar pesos guardados por el propio notebook
model = CNN()
model.load_state_dict(torch.load('modelo_cifar10.pth', map_location='cpu', weights_only=True))
model.eval()

# Mostrar una imagen aleatoria y comparar predicción con etiqueta real
def mostrar_y_predecir_imagen(testloader):
    dataiter = iter(testloader)
    images, labels = next(dataiter)

    indice_aleatorio = np.random.randint(0, images.size(0))
    imagen_aleatoria = images[indice_aleatorio].unsqueeze(0)
    etiqueta_real = labels[indice_aleatorio].item()

    with torch.no_grad():
        salida = model(imagen_aleatoria)
        _, prediccion = torch.max(salida, 1)

    plt.imshow(np.transpose(imagen_aleatoria[0].numpy(), (1, 2, 0)) / 2 + 0.5)
    plt.title(f'Predicción: {clases[prediccion.item()]}, Real: {clases[etiqueta_real]}')
    plt.axis('off')
    plt.show()

mostrar_y_predecir_imagen(testloader)


## Discussion and limitations
The original diploma notebook recorded 77.29% test accuracy; that result is historical and was not independently reproduced for this publication. The original evaluation applied random augmentation to the test set; this public version changes that to deterministic test preprocessing. Training can be time-consuming on a CPU. This is an educational project, not a production-ready classifier.